# Vision Transformer (ViT) — Paper-to-Code Mock (Colab)

**Paper:** An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale (Dosovitskiy et al., 2020) — https://arxiv.org/abs/2010.11929

Medium mock (~40 min). Fill in the `ViT` stub (patch embed -> CLS -> positions -> encoder -> head), run the quadrant-classification demo, then the sanity checks. Reuses the multi-head attention from the Attention mock. Reference solution in the last cell.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)


# Provided: reused multi-head self-attention (see the Attention mock).
def scaled_dot_product_attention(q, k, v):
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    attn = scores.softmax(dim=-1)
    return attn @ v, attn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads, self.d_k = d_model, n_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        q, k, v = self._split(self.wq(x)), self._split(self.wk(x)), self._split(self.wv(x))
        out, attn = scaled_dot_product_attention(q, k, v)
        B, _, T, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.wo(out), attn


class EncoderBlock(nn.Module):
    """Pre-norm Transformer encoder block."""
    def __init__(self, dim, n_heads, mlp_ratio=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, n_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio), nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        a, attn = self.attn(self.norm1(x))
        x = x + a
        x = x + self.mlp(self.norm2(x))
        return x, attn

## 1. Implement the ViT pipeline (YOUR TASK)

Build: patch embed (Conv2d with `kernel=stride=patch`) -> prepend learnable CLS token -> add learnable positional embeddings -> run the encoder blocks -> classify from the CLS token. Number of patches: `(img_size/patch)**2`.

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size=12, patch=4, in_ch=1, dim=32, depth=2,
                 n_heads=4, n_classes=4):
        super().__init__()
        assert img_size % patch == 0
        self.n_patches = (img_size // patch) ** 2
        # TODO: self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
        # TODO: self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        # TODO: self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches + 1, dim))
        # TODO: self.blocks = nn.ModuleList([EncoderBlock(dim, n_heads) for _ in range(depth)])
        # TODO: self.norm = nn.LayerNorm(dim); self.head = nn.Linear(dim, n_classes)
        # TODO: init cls_token and pos_embed with std=0.02
        raise NotImplementedError("fill me in")

    def patchify(self, imgs):
        # TODO: conv -> (B, dim, H/p, W/p); flatten(2).transpose(1,2) -> (B, n_patches, dim)
        raise NotImplementedError("fill me in")

    def forward(self, imgs, return_attn=False):
        # TODO: patchify -> prepend CLS (expand to batch) -> add pos_embed
        # TODO: run blocks (keep last attn) -> final LN -> head on x[:, 0]
        raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (which-quadrant visual rule)
Tiny 12x12 grayscale images, patch 4 -> 9 patches. A 3x3 bright square sits in one of 4 quadrants; label = which quadrant. Solving it needs positional info + global attention. Train acc/test acc should climb well above the 0.25 chance level.

In [ ]:
def make_data(n, img_size=12, seed=0):
    g = torch.Generator().manual_seed(seed)
    imgs = 0.1 * torch.randn(n, 1, img_size, img_size, generator=g)
    labels = torch.randint(0, 4, (n,), generator=g)
    half = img_size // 2
    for i in range(n):
        q = labels[i].item()
        r0 = 0 if q in (0, 1) else half      # top for 0,1 / bottom for 2,3
        c0 = 0 if q in (0, 2) else half      # left for 0,2 / right for 1,3
        rr = r0 + torch.randint(0, half - 2, (1,), generator=g).item()
        cc = c0 + torch.randint(0, half - 2, (1,), generator=g).item()
        imgs[i, 0, rr:rr + 3, cc:cc + 3] += 3.0
    return imgs, labels


torch.manual_seed(0)
Xtr, ytr = make_data(600, seed=1)
Xte, yte = make_data(300, seed=2)
model = ViT(img_size=12, patch=4, dim=32, depth=2, n_heads=4, n_classes=4)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

model.train()
for epoch in range(40):
    perm = torch.randperm(Xtr.size(0))
    for i in range(0, Xtr.size(0), 64):
        idx = perm[i:i + 64]
        loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    tr_acc = (model(Xtr).argmax(1) == ytr).float().mean().item()
    te_acc = (model(Xte).argmax(1) == yte).float().mean().item()
print(f"train acc {tr_acc:.3f}   test acc {te_acc:.3f}   (chance = 0.25)")
# NOTE: this only shows the pipeline LEARNS. The real ViT benefit (beating CNNs
# as pretraining data scales) needs ImageNet-21k/JFT-300M and can't be shown at toy scale;
# ViT's known weakness is exactly that it is data hungry.

## 3. Sanity checks

In [ ]:
m = ViT(img_size=12, patch=4, dim=32, depth=2, n_heads=4, n_classes=4)
imgs = torch.randn(8, 1, 12, 12)

# 1) patchify -> right number of patches + embedded shape (B, n_patches, dim)
p = m.patchify(imgs)
assert m.n_patches == (12 // 4) * (12 // 4) == 9
assert p.shape == (8, 9, 32)

# 2) after adding CLS, sequence length = n_patches + 1
seq = torch.cat([m.cls_token.expand(8, -1, -1), p], dim=1)
assert seq.shape[1] == m.n_patches + 1 == 10

# 3) positional embedding shape matches
assert m.pos_embed.shape == (1, m.n_patches + 1, 32)

# 4) output logits shape == (B, n_classes)
assert m(imgs).shape == (8, 4)

# 5) trained test accuracy beats chance (the demonstration)
model.eval()
with torch.no_grad():
    te = (model(Xte).argmax(1) == yte).float().mean().item()
assert te > 0.6, te                      # chance = 0.25

# 6) attention rows sum to 1 + grads reach patch-embed, CLS, pos-embed
m.train()
logits, attn = m(imgs, return_attn=True)
assert torch.allclose(attn.sum(-1), torch.ones_like(attn.sum(-1)), atol=1e-5)
assert (attn >= 0).all()
logits.sum().backward()
assert m.proj.weight.grad.abs().sum() > 0
assert m.cls_token.grad.abs().sum() > 0
assert m.pos_embed.grad.abs().sum() > 0

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class ViTSolution(nn.Module):
    def __init__(self, img_size=12, patch=4, in_ch=1, dim=32, depth=2,
                 n_heads=4, n_classes=4):
        super().__init__()
        assert img_size % patch == 0
        self.n_patches = (img_size // patch) ** 2
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches + 1, dim))
        self.blocks = nn.ModuleList([EncoderBlock(dim, n_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.pos_embed, std=0.02)

    def patchify(self, imgs):
        x = self.proj(imgs)
        return x.flatten(2).transpose(1, 2)

    def forward(self, imgs, return_attn=False):
        B = imgs.size(0)
        x = self.patchify(imgs)
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        x = x + self.pos_embed
        last_attn = None
        for blk in self.blocks:
            x, last_attn = blk(x)
        x = self.norm(x)
        logits = self.head(x[:, 0])
        return (logits, last_attn) if return_attn else logits